# IF Model training, hyperparameter tuning and evaluation
In this section, several machine learning classifiers are trained and evaluated to predict match outcomes (blueWins) based on the 'cleaned' feature set.
- Unnesecary columns are removed from the dataset, the target variable is defined and the features. Data is split into training (85%) and validation (15%) using stratified sampling.
- The models that are used in this block are; Logistic regression, SVM, kNN, Decision tree, Random forest, Gradient boostingn and Naïve bayes
- 5 fold cross validation is performed for hyperparameter tuning using 'GridSearchCV'
- Each model, with its best found parameters, is evaluated on the validation set to estimate the generalized performance of the model

In [ ]:
# Drop unnecessary columns
X = cleaned_imputed_df_IF.drop(columns=['blueWins','outlier_flag','outlier_score',"PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster"], errors='ignore')
y = cleaned_imputed_df_IF['blueWins']

# Split into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define models with parameter grids
model_grids = {
    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "clf__C": [0.01, 0.1, 1, 10],
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"]
        }
    },
    "SVM (RBF kernel)": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True))
        ]),
        "params": {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", "auto"]
        }
    },
    "k-NN": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9],
            "clf__weights": ["uniform", "distance"]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.1],
            "max_depth": [3, 5]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {}  # No hyperparameters to tune
    }
}

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc_macro": "roc_auc_ovr"
}

# Hyperparameter tuning & CV
results = []

for name, mg in model_grids.items():
    model = mg["model"]
    param_grid = mg["params"]
    
    if param_grid:  # only tune if grid exists
        search = GridSearchCV(model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
    else:
        # No tuning for Naive Bayes
        best_model = model
        best_model.fit(X_train, y_train)
        best_params = {}
    
    scores = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Best Params": best_params,
        "Accuracy mean (CV)": scores["test_accuracy"].mean(),
        "F1 macro mean (CV)": scores["test_f1_macro"].mean(),
        "ROC-AUC mean (CV)": scores["test_roc_auc_macro"].mean()
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy mean (CV)", ascending=False).reset_index(drop=True)
print("=== Cross-validation results with hyperparameter optimization ===")
print(results_df)

# Fit all models with best hyperparameters on the full training set
fitted_models = {}

for name, mg in model_grids.items():
    model = mg["model"]
    param_grid = mg["params"]

    if param_grid:
        # Tune hyperparameters with GridSearchCV
        search = GridSearchCV(model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
    else:
        # No hyperparameters to tune (e.g., Naive Bayes)
        best_model = model
        best_model.fit(X_train, y_train)

    fitted_models[name] = best_model

# Evaluate all models on the validation set
val_results = []

for name, model in fitted_models.items():
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None

    val_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "F1 macro": f1_score(y_val, y_val_pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else np.nan
    })

val_results_df = pd.DataFrame(val_results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("=== Validation set performance for all models ===")
print(val_results_df)

# Visualization of hyperparameter tuning
- In this block a function is defined that will perform the GirdSearchCV, extract its results and ranks all parameter combinations by the mean cross-validation accuracy.
- The top performing settings are then visualized with simple bar plots to have a clear overview of which settings are good to use.

In [ ]:
# Full hyperparameter visualization setup

def plot_top_hyperparameter_results(model_name, model, param_grid, X_train, y_train, cv, top_n=10, save=False):
    """
    Runs GridSearchCV and plots the top N hyperparameter combinations by mean CV accuracy.
    Works for any number of hyperparameters.

    Parameters:
    - model_name: str, name of the model (for title and filename)
    - model: sklearn estimator
    - param_grid: dict, hyperparameter grid
    - X_train, y_train: training data
    - cv: cross-validation folds
    - top_n: number of top combinations to plot
    - save: if True, saves each plot as a PNG file
    """
    if not param_grid:
        print(f"No hyperparameters to tune for {model_name}.")
        return

    print(f"\n=== Running GridSearchCV for {model_name} ===")
    search = GridSearchCV(model, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    search.fit(X_train, y_train)

    # Extract and process results
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values(by='mean_test_score', ascending=False).head(top_n).reset_index(drop=True)

    # Build readable parameter labels for each combination
    cv_results['params_str'] = cv_results.apply(
        lambda row: ", ".join([
            f"{col.replace('param_', '')}={row[col]}" for col in cv_results.columns if col.startswith('param_')
        ]), axis=1
    )

    # Plot top configurations
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=cv_results,
        x='mean_test_score',
        y='params_str',
        palette='YlGnBu'
    )
    plt.xlabel("Mean CV Accuracy")
    plt.ylabel("Hyperparameter Combination")
    plt.title(f"Top {top_n} Hyperparameter Configurations: {model_name}")
    plt.tight_layout()

    # Optionally save the figure
    if save:
        filename = f"top_{top_n}_hyperparams_{model_name.replace(' ', '_').lower()}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved plot as {filename}")

    plt.show()


def run_top_hyperparameter_plots(model_grids, X_train, y_train, cv, top_n=10, save=False):
    """
    Loops through all models in model_grids and plots top hyperparameter results.
    """
    for name, mg in model_grids.items():
        plot_top_hyperparameter_results(
            model_name=name,
            model=mg["model"],
            param_grid=mg["params"],
            X_train=X_train,
            y_train=y_train,
            cv=cv,
            top_n=top_n,
            save=save
        )


# Visualize all models
run_top_hyperparameter_plots(model_grids, X_train, y_train, cv, top_n=10, save=False)

# Isolation forest using no hyperparameter optimization (for comparison)
- This codeblock shows the ML without hyperparameter optimization to just compare the performances with the codeblock above.

In [ ]:
# Drop unnecessary columns
X = cleaned_imputed_df_IF.drop(columns=['blueWins','outlier_flag','outlier_score',"PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster"], errors='ignore')
y = cleaned_imputed_df_IF['blueWins']

# Split into train/test/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 5-fold stratified CV on training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "SVM (RBF kernel)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True))
    ]),
    "k-NN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier())
    ]),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc_macro": "roc_auc_ovr"
}

results = []

# Cross-validate on training data
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy mean (CV)": scores["test_accuracy"].mean(),
        "F1 macro mean (CV)": scores["test_f1_macro"].mean(),
        "ROC-AUC mean (CV)": scores["test_roc_auc_macro"].mean(),
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy mean (CV)", ascending=False).reset_index(drop=True)
print("=== Cross-validation results ===")
print(results_df)

# Pick best model (e.g. top performer)
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred = best_model.predict(X_val)
y_val_proba = best_model.predict_proba(X_val)[:, 1] if hasattr(best_model, "predict_proba") else None

val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro")
val_auc = roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else np.nan

print(f"\n=== Validation performance ({best_model_name}) ===")
print(f"Accuracy: {val_acc:.3f}")
print(f"F1 macro: {val_f1:.3f}")
print(f"ROC-AUC: {val_auc:.3f}")

# kNN Model training, hyperparameter tuning and evaluation
In this section, several machine learning classifiers are trained and evaluated to predict match outcomes (blueWins) based on the 'cleaned' feature set.
- Unnesecary columns are removed from the dataset, the target variable is defined and the features. Data is split into training (85%) and validation (15%) using stratified sampling.
- The models that are used in this block are; Logistic regression, SVM, kNN, Decision tree, Random forest, Gradient boostingn and Naïve bayes
- 5 fold cross validation is performed for hyperparameter tuning using 'GridSearchCV'
- Each model, with its best found parameters, is evaluated on the validation set to estimate the generalized performance of the model
The code is basically the same as for the IF multivariate outlier detection method but this time it works with the kNN dataset.

In [ ]:
# Drop unnecessary columns
X = cleaned_imputed_df_knn.drop(columns=[
    'blueWins', 'gameId','outlier_flag','outlier_score',
    "PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster",
    'knn_outlier_flag','knn_outlier_score'
], errors='ignore')
y = cleaned_imputed_df_knn['blueWins']

# Split into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define models with parameter grids
model_grids = {
    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "clf__C": [0.01, 0.1, 1, 10],
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"]
        }
    },
    "SVM (RBF kernel)": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True))
        ]),
        "params": {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", "auto"]
        }
    },
    "k-NN": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9],
            "clf__weights": ["uniform", "distance"]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.1],
            "max_depth": [3, 5]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {}  # No hyperparameters
    }
}

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc_macro": "roc_auc_ovr"
}

# Hyperparameter tuning & cross-validation
cv_results = []
fitted_models = {}

for name, mg in model_grids.items():
    model = mg["model"]
    param_grid = mg["params"]

    if param_grid:
        search = GridSearchCV(model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
    else:
        best_model = model
        best_model.fit(X_train, y_train)
        best_params = {}

    # CV performance
    scores = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    cv_results.append({
        "Model": name,
        "Best Params": best_params,
        "Accuracy mean (CV)": scores["test_accuracy"].mean(),
        "F1 macro mean (CV)": scores["test_f1_macro"].mean(),
        "ROC-AUC mean (CV)": scores["test_roc_auc_macro"].mean()
    })

    # Store fitted model for validation evaluation
    fitted_models[name] = best_model

cv_results_df = pd.DataFrame(cv_results).sort_values(by="Accuracy mean (CV)", ascending=False).reset_index(drop=True)
print("=== Cross-validation results with hyperparameter optimization ===")
print(cv_results_df)

# Evaluate all models on the validation set
val_results = []

for name, model in fitted_models.items():
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None

    val_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "F1 macro": f1_score(y_val, y_val_pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else np.nan
    })

val_results_df = pd.DataFrame(val_results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("=== Validation set performance for all models ===")
print(val_results_df)

# Visualization of hyperparameter tuning
- In this block a function is defined that will perform the GirdSearchCV, extract its results and ranks all parameter combinations by the mean cross-validation accuracy.
- The top performing settings are then visualized with simple bar plots to have a clear overview of which settings are good to use.
This is the same code for visualization used but this time for the kNN outlier detection dataset.

In [ ]:
# Full hyperparameter visualization setup

def plot_top_hyperparameter_results(model_name, model, param_grid, X_train, y_train, cv, top_n=10, save=False):
    """
    Runs GridSearchCV and plots the top N hyperparameter combinations by mean CV accuracy.
    Works for any number of hyperparameters.

    Parameters:
    - model_name: str, name of the model (for title and filename)
    - model: sklearn estimator
    - param_grid: dict, hyperparameter grid
    - X_train, y_train: training data
    - cv: cross-validation folds
    - top_n: number of top combinations to plot
    - save: if True, saves each plot as a PNG file
    """
    if not param_grid:
        print(f"No hyperparameters to tune for {model_name}.")
        return

    print(f"\n=== Running GridSearchCV for {model_name} ===")
    search = GridSearchCV(model, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    search.fit(X_train, y_train)

    # Extract and process results
    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results.sort_values(by='mean_test_score', ascending=False).head(top_n).reset_index(drop=True)

    # Build readable parameter labels for each combination
    cv_results['params_str'] = cv_results.apply(
        lambda row: ", ".join([
            f"{col.replace('param_', '')}={row[col]}" for col in cv_results.columns if col.startswith('param_')
        ]), axis=1
    )

    # Plot top configurations
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=cv_results,
        x='mean_test_score',
        y='params_str',
        palette='YlGnBu'
    )
    plt.xlabel("Mean CV Accuracy")
    plt.ylabel("Hyperparameter Combination")
    plt.title(f"Top {top_n} Hyperparameter Configurations: {model_name}")
    plt.tight_layout()

    # Optionally save the figure
    if save:
        filename = f"top_{top_n}_hyperparams_{model_name.replace(' ', '_').lower()}.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved plot as {filename}")

    plt.show()


def run_top_hyperparameter_plots(model_grids, X_train, y_train, cv, top_n=10, save=False):
    """
    Loops through all models in model_grids and plots top hyperparameter results.
    """
    for name, mg in model_grids.items():
        plot_top_hyperparameter_results(
            model_name=name,
            model=mg["model"],
            param_grid=mg["params"],
            X_train=X_train,
            y_train=y_train,
            cv=cv,
            top_n=top_n,
            save=save
        )


# Visualize all models
run_top_hyperparameter_plots(model_grids, X_train, y_train, cv, top_n=10, save=False)

# kNN using no hyperparameter optimization (for comparison)
- This codeblock shows the ML without hyperparameter optimization to just compare the performances with the codeblock above.

In [ ]:
# Drop unnecessary columns
X = cleaned_imputed_df_knn.drop(columns=['blueWins', 'gameId','outlier_flag','outlier_score',"PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster",'knn_outlier_flag','knn_outlier_score'], errors='ignore')
y = cleaned_imputed_df_knn['blueWins']

# Split into train/test/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# define models inside pipelines where needed
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "SVM (RBF kernel)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True))  # probability=True so we can compute ROC-AUC
    ]),
    "k-NN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier())
    ]),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc_macro": "roc_auc_ovr"
}

results = []

# Cross-validate on training data
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy mean (CV)": scores["test_accuracy"].mean(),
        "F1 macro mean (CV)": scores["test_f1_macro"].mean(),
        "ROC-AUC mean (CV)": scores["test_roc_auc_macro"].mean(),
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy mean (CV)", ascending=False).reset_index(drop=True)
print("=== Cross-validation results ===")
print(results_df)

# Pick best model (e.g. top performer)
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred = best_model.predict(X_val)
y_val_proba = best_model.predict_proba(X_val)[:, 1] if hasattr(best_model, "predict_proba") else None

val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro")
val_auc = roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else np.nan

print(f"\n=== Validation performance ({best_model_name}) ===")
print(f"Accuracy: {val_acc:.3f}")
print(f"F1 macro: {val_f1:.3f}")
print(f"ROC-AUC: {val_auc:.3f}")

# Model training, hyperparameter tuning and evaluation
In this section, several machine learning classifiers are trained and evaluated to predict match outcomes (blueWins) based on the 'cleaned' feature set.
- Unnesecary columns are removed from the dataset, the target variable is defined and the features. Data is split into training (85%) and validation (15%) using stratified sampling.
- The models that are used in this block are; Logistic regression, SVM, kNN, Decision tree, Random forest, Gradient boostingn and Naïve bayes
- 5 fold cross validation is performed for hyperparameter tuning using 'GridSearchCV'
- Each model, with its best found parameters, is evaluated on the validation set to estimate the generalized performance of the model
The code is (again) basically the same as the previous ML frameworks but the big difference is that this uses the dirty data without any multivariate outlier detection methods.

In [ ]:
# Define features (X) and target (y)
# Drop identifiers and the target from X
X = imputed_df_dirty.drop(columns=['blueWins', 'gameId','outlier_flag','outlier_score',"PCA1", "PCA2", "tSNE1", "tSNE2", "Cluster",'knn_outlier_flag','knn_outlier_score'], errors='ignore')
y = data['blueWins']  # target column still comes from the original data

# Split into train/test/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define models with parameter grids
model_grids = {
    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "clf__C": [0.01, 0.1, 1, 10],
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"]
        }
    },
    "SVM (RBF kernel)": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True))
        ]),
        "params": {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", "auto"]
        }
    },
    "k-NN": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9],
            "clf__weights": ["uniform", "distance"]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.01, 0.1],
            "max_depth": [3, 5]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {}  # No hyperparameters
    }
}

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc_macro": "roc_auc_ovr"
}

# Hyperparameter tuning & cross-validation
cv_results = []
fitted_models = {}

for name, mg in model_grids.items():
    model = mg["model"]
    param_grid = mg["params"]

    if param_grid:
        search = GridSearchCV(model, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
    else:
        best_model = model
        best_model.fit(X_train, y_train)
        best_params = {}

    # CV performance
    scores = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    cv_results.append({
        "Model": name,
        "Best Params": best_params,
        "Accuracy mean (CV)": scores["test_accuracy"].mean(),
        "F1 macro mean (CV)": scores["test_f1_macro"].mean(),
        "ROC-AUC mean (CV)": scores["test_roc_auc_macro"].mean()
    })

    # Store fitted model for validation evaluation
    fitted_models[name] = best_model

cv_results_df = pd.DataFrame(cv_results).sort_values(by="Accuracy mean (CV)", ascending=False).reset_index(drop=True)
print("=== Cross-validation results with hyperparameter optimization ===")
print(cv_results_df)

# Evaluate all models on the validation set
val_results = []

for name, model in fitted_models.items():
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None

    val_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "F1 macro": f1_score(y_val, y_val_pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_val, y_val_proba) if y_val_proba is not None else np.nan
    })

val_results_df = pd.DataFrame(val_results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("=== Validation set performance for all models ===")
print(val_results_df)